# Введение #

В этом уроке мы увидим, как можно создавать нейронные сети, способные обучаться сложным взаимосвязям, которыми знамениты глубокие нейросети.

Ключевая идея здесь — *модульность*: построение сложной сети из более простых функциональных блоков. Мы уже видели, как линейный блок вычисляет линейную функцию — теперь мы разберемся, как комбинировать и модифицировать эти одиночные блоки, чтобы моделировать более сложные зависимости.

# Слои #

Нейронные сети обычно организуют свои нейроны в **слои**. Когда мы объединяем линейные блоки с общим набором входных данных, мы получаем **полносвязный** (dense) слой.

<figure style="padding: 1em;">
<img src="https://storage.googleapis.com/kaggle-media/learn/images/2MA4iMV.png" width="300" alt="Стек из трех кругов во входном слое, соединенных с двумя кругами в полносвязном слое.">
<figcaption style="textalign: center; font-style: italic"><center>Полносвязный слой из двух линейных блоков, получающих два входа и смещение.
</center></figcaption>
</figure>

Можно представить, что каждый слой в нейронной сети выполняет своего рода относительно простое преобразование. Через глубокий стек слоев нейронная сеть может преобразовывать свои входные данные всё более и более сложными способами. В хорошо обученной нейронной сети каждый слой представляет собой преобразование, которое немного приближает нас к решению.

<blockquote style="margin-right:auto; margin-left:auto; background-color: #ebf9ff; padding: 1em; margin:24px;">
    <strong>Множество видов слоев</strong><br>
«Слой» в Keras — это очень общее понятие. Слой может быть, по сути, любым видом <em>преобразования данных</em>. Многие слои, такие как <a href="https://www.tensorflow.org/api_docs/python/tf/keras/layers/Conv2D">сверточные</a> и <a href="https://www.tensorflow.org/api_docs/python/tf/keras/layers/RNN">рекуррентные</a> слои, преобразуют данные с помощью нейронов и отличаются прежде всего структурой связей, которые они формируют. Другие же используются для <a href="https://www.tensorflow.org/api_docs/python/tf/keras/layers/Embedding">проектирования признаков</a> (feature engineering) или просто для <a href="https://www.tensorflow.org/api_docs/python/tf/keras/layers/Add">простых арифметических операций</a>. Есть целый мир слоев, которые стоит изучить — <a href="https://www.tensorflow.org/api_docs/python/tf/keras/layers">посмотрите их здесь</a>!
</blockquote>

# Функция активации #

Однако оказалось, что два полносвязных слоя, расположенных друг за другом без чего-либо между ними, ничем не лучше одного одиночного полносвязного слоя. Сами по себе полносвязные слои никогда не выведут нас за пределы мира линий и плоскостей. Нам нужно что-то *нелинейное*. Нам нужны функции активации.

<figure style="padding: 1em;">
<img src="https://storage.googleapis.com/kaggle-media/learn/images/OLSUEYT.png" width="400" alt=" ">
<figcaption style="textalign: center; font-style: italic"><center>Без функций активации нейронные сети могут изучать только линейные зависимости. Чтобы подбирать кривые, нам потребуется использовать функции активации. 
</center></figcaption>
</figure>

**Функция активации** — это просто функция, которую мы применяем к каждому выходу слоя (его *активациям*). Самая распространенная из них — функция *выпрямления* (rectifier) $max(0, x)$.

<figure style="padding: 1em;">
<img src="https://storage.googleapis.com/kaggle-media/learn/images/aeIyAlF.png" width="400" alt="График функции выпрямления. Линия y=x при x>0 и y=0 при x<0, образуя форму «шарнира» типа '_/'.">
<figcaption style="textalign: center; font-style: italic"><center>
</center></figcaption>
</figure>

График функции выпрямления представляет собой линию, в которой отрицательная часть «выпрямлена» до нуля. Применение этой функции к выходам нейрона создает *излом* в данных, уводя нас от простых прямых.

Когда мы присоединяем функцию выпрямления к линейному блоку, мы получаем **выпрямленный линейный блок**, или **ReLU**. (По этой причине функцию выпрямления часто называют «функцией ReLU»). Применение активации ReLU к линейному блоку означает, что выход становится `max(0, w * x + b)`, что на схеме можно изобразить так:

<figure style="padding: 1em;">
<img src="https://storage.googleapis.com/kaggle-media/learn/images/eFry7Yu.png" width="250" alt="Диаграмма одного ReLU. Похоже на линейный блок, но вместо символа «+» теперь изображен «шарнир» '_/'. ">
<figcaption style="textalign: center; font-style: italic"><center>Выпрямленный линейный блок (ReLU).
</center></figcaption>
</figure>

# Стек полносвязных слоев #

Теперь, когда у нас появилась нелинейность, давайте посмотрим, как мы можем складывать слои в стек, чтобы получить сложные преобразования данных.

<figure style="padding: 1em;">
<img src="https://storage.googleapis.com/kaggle-media/learn/images/Y5iwFQZ.png" width="450" alt="Входной слой, два скрытых слоя и финальный линейный слой.">
<figcaption style="textalign: center; font-style: italic"><center>Стек полносвязных слоев образует «полносвязную» сеть.
</center></figcaption>
</figure>

Слои, расположенные перед выходным слоем, иногда называют **скрытыми**, так как мы никогда не видим их выходы напрямую.

Заметьте, что финальный (выходной) слой является линейным блоком (то есть без функции активации). Это делает данную сеть подходящей для задачи регрессии, где мы пытаемся предсказать произвольное числовое значение. Другие задачи (например, классификация) могут потребовать наличия функции активации на выходе.

## Создание последовательных моделей (Sequential Models) ##

Модель `Sequential`, которую мы использовали, соединяет список слоев по порядку от первого до последнего: первый слой получает входные данные, последний слой выдает результат. Это создает модель, изображенную на рисунке выше:

In [1]:
from tensorflow import keras
from tensorflow.keras import layers

model = keras.Sequential([
    # скрытые слои ReLU
    layers.Dense(units=4, activation='relu', input_shape=[2]),
    layers.Dense(units=3, activation='relu'),
    # линейный выходной слой 
    layers.Dense(units=1),
])

c:\Users\iOlvik\Documents\1_Dev\Work\ML_training\venv\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Обязательно передавайте все слои вместе в одном списке, например `[layer, layer, layer, ...]`, а не в качестве отдельных аргументов. Чтобы добавить функцию активации к слою, просто укажите её название в аргументе `activation`.

# Ваша очередь #

Теперь [**создайте глубокую нейронную сеть**](https://www.kaggle.com/kernels/fork/11887344) для набора данных *Concrete* (Бетон).

---




*Есть вопросы или комментарии? Посетите [форум обсуждения курса](https://www.kaggle.com/learn/intro-to-deep-learning/discussion), чтобы пообщаться с другими студентами.*